# 03 · Classificação (probabilidades)

**O que esta etapa faz:** para cada carta e ano, treina um Random Forest com amostras da super carta (urbanas e não urbanas, só do conjunto de treino) sobre o mosaico anual, e calcula a probabilidade de 4 classes: **class1_prob** urbano, **class2_prob** vegetação, **class3_prob** água, **class4_prob** agropecuária (0-100).

A probabilidade urbana alimenta a mancha urbana. A de vegetação alimenta a vegetação intraurbana.

**Equivalente na Col 11 (versão de produção):** `collection 11/codes_producao/02_classificacao/`: `classification_batch_v3.js` e `class_lib.js`.

**Parâmetros** em `config/col12.yaml`, bloco `classification`: número de amostras, proporção não urbano/urbano, árvores e formato de saída:
- `per_year`: uma imagem por carta e ano (como na Col 11);
- `multiband`: uma imagem por carta com todos os anos (`class1_prob_1985` ... `class4_prob_2025`).

Nada é exportado sem você rodar explicitamente o bloco 4.

## 0 · Preparação

In [ ]:
import sys, os
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    !git clone -q -b col12 https://github.com/jpedrassoli/brazil-urban-area.git repo 2>/dev/null || (cd repo && git pull -q)
    %pip install -q -e "repo/lulc_30m_landsat/collection 12" geemap
    RAIZ = 'repo/lulc_30m_landsat/collection 12'
else:
    RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

import ee
from mb_urban import auth, config, grids, mosaic, samples, classify, tasks, col11
cfg = config.load()

## 1 · Conta do Earth Engine
Usa o login padrão do Earth Engine. No Colab, abre a autorização da sua conta.

In [ ]:
CONTA = 'default'
if CONTA == 'default':
    ee.Authenticate()   # só pede login se ainda não houver credencial válida
auth.init(CONTA)
print('projeto:', cfg['cloud_project'], '| destino:', config.destination_root())

## 2 · Parâmetros

In [ ]:
CARTA = 'SF-23-Y-C'
ANOS = config.years(cfg)            # ou uma lista menor, ex.: [1985, 2005, 2020]
FORMATO = cfg['classification']['output']
print(CARTA, f'{ANOS[0]}-{ANOS[-1]}', FORMATO, '| mosaico:', cfg['mosaic']['source'], cfg['mosaic']['encoding'])

## 3 · Amostras de treino de um ano
Confere quantas amostras de cada classe entram no treino.

In [ ]:
ANO_TESTE = ANOS[-1]
treino = classify.training_samples(ANO_TESTE, grids.context(CARTA, cfg), cfg)
print('janela:', classify.window_for_year(ANO_TESTE), '| por classe:', treino.aggregate_histogram('class').getInfo())

## 4 · Exportar
Cria as tarefas e envia respeitando o limite de tarefas simultâneas da conta. Assets que já existem são pulados.
**Só envia quando `EXPORTAR = True`.**

In [ ]:
EXPORTAR = False
if EXPORTAR:
    specs = classify.export_tasks(CARTA, years=ANOS, output=FORMATO, cfg=cfg)
    print(tasks.run_queue(specs, account=CONTA, log_name='classificacao', cfg=cfg))
else:
    print('exportação desligada (EXPORTAR = False)')

## 5 · Acompanhar tarefas
Rode de novo para atualizar.

In [ ]:
tasks.my_tasks(CONTA)

## 6 · Comparar com a Col 11
Depois que a exportação terminar, compara a probabilidade urbana com a da Col 11 na mesma carta e ano: diferença média e máxima, e um mapa da diferença.

In [ ]:
ANO_CMP = 2020
nome = f'prob_{CARTA}' if FORMATO == 'multiband' else f'prob_{CARTA}_{ANO_CMP}'
img = ee.Image(config.asset_path(cfg['classification']['folder'], nome, cfg=cfg))
banda = f'class1_prob_{ANO_CMP}' if FORMATO == 'multiband' else 'class1_prob'
nova = img.select(banda).rename('nova')
antiga = col11.probability(CARTA, ANO_CMP).select('class1_prob').rename('col11')
dif = nova.subtract(antiga).abs().rename('dif')
stats = dif.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.max(), '', True), grids.region(CARTA, cfg), 300, maxPixels=1e10)
print('diferença absoluta (amostrada a 300 m):', stats.getInfo())

import geemap
m = geemap.Map(); m.centerObject(grids.region(CARTA, cfg), 9)
m.addLayer(nova, {'min': 0, 'max': 100}, 'Col 12', False)
m.addLayer(antiga, {'min': 0, 'max': 100}, 'Col 11', False)
m.addLayer(dif, {'min': 0, 'max': 30, 'palette': ['white', 'red']}, 'diferença')
m